# Experiment: CytoRAG Graph RAG Kaggle Experiment

Objective:
- использовать результаты `experiments_res.md` как подспорье, не изменяя существующие ноутбуки;
- взять лучшую embedding-модель для полного RAG-цикла: `cointegrated/rubert-tiny2`;
- проверить лучшие chunking-стратегии из отчета: `sentence_window_2`, `structural_then_sentence_window_2`, `structural_subcase`;
- построить Graph RAG поверх гибридного поиска и замерить те же метрики: `Hit Rate`, `MRR`, `context_precision`, `context_recall`, `faithfulness_ragas`, `answer_relevancy`.

Notebook рассчитан на Kaggle. Быстрый прогон retrieval работает без LLM. Для генерации ответов и RAGAS включи `RUN_LOCAL_LLM=True` и `RUN_RAGAS=True` в конфигурации.

In [ ]:
# Kaggle setup for the retrieval part. Run this cell once, then restart the Kaggle session/kernel.
# Dependencies are installed into an isolated folder, not into Kaggle's global Python environment.
import shutil
shutil.rmtree("/kaggle/working/cytorag_deps", ignore_errors=True)
%pip install -q --target /kaggle/working/cytorag_deps --upgrade --ignore-installed --no-cache-dir --no-deps "transformers==4.44.2" "tokenizers==0.19.1" "huggingface-hub==0.34.4" "safetensors>=0.4.3" "accelerate==0.33.0" "faiss-cpu>=1.8,<2" "rank-bm25==0.2.2" "networkx>=3.2,<4"
print("Retrieval setup complete. IMPORTANT: restart the Kaggle session/kernel now, then run from the imports cell.")


In [ ]:
# Optional RAGAS setup. Run this cell only if the retrieval experiment works and you set RUN_RAGAS=1.
# These packages also go into /kaggle/working/cytorag_deps to avoid global Kaggle dependency conflicts.
%pip install -q --target /kaggle/working/cytorag_deps --upgrade --ignore-installed --no-cache-dir "transformers==4.44.2" "tokenizers==0.19.1" "huggingface-hub==0.34.4" "ragas>=0.2,<0.5" "datasets>=2.19,<5" "langchain-huggingface>=0.1,<1" "langchain-core>=0.2,<1" "langchain>=0.2,<1"
print("RAGAS setup complete. Restart the Kaggle session/kernel, then rerun from the imports/config cell.")


In [ ]:
from __future__ import annotations

import gc
import json
import os
import random
import re
import sys
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("WANDB_DISABLED", "true")
os.environ.setdefault("DISABLE_MLFLOW_INTEGRATION", "true")
CYTORAG_DEPS_DIR = Path(os.getenv("CYTORAG_DEPS_DIR", "/kaggle/working/cytorag_deps"))
if CYTORAG_DEPS_DIR.exists() and str(CYTORAG_DEPS_DIR) not in sys.path:
    sys.path.insert(0, str(CYTORAG_DEPS_DIR))
    print(f"Using isolated dependencies from: {CYTORAG_DEPS_DIR}")

import faiss
import networkx as nx
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from rank_bm25 import BM25Okapi
from transformers import AutoModel, AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer, pipeline

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
warnings.filterwarnings("ignore")

def str_to_bool(value: str, default: bool = False) -> bool:
    if value is None:
        return default
    return str(value).strip().lower() in {"1", "true", "yes", "y", "on"}

def resolve_base_dir() -> Path:
    env_data_path = os.getenv("CYTORAG_DATA_PATH")
    if env_data_path:
        data_path = Path(env_data_path).expanduser()
        if data_path.exists() and data_path.name == "bethesda_ground_truth.json":
            return data_path.parent
        if (data_path / "bethesda_ground_truth.json").exists():
            return data_path

    cwd = Path.cwd()
    direct_candidates = [
        cwd,
        cwd / "CytoRAG",
        Path("/kaggle/input/cytorag"),
        Path("/kaggle/input/cytorag/CytoRAG"),
        Path("/kaggle/working"),
        Path("/kaggle/working/CytoRAG"),
    ]
    for candidate in direct_candidates:
        if (candidate / "bethesda_ground_truth.json").exists():
            return candidate

    search_roots = [cwd, Path("/kaggle/input"), Path("/kaggle/working")]
    for root in search_roots:
        if not root.exists():
            continue
        matches = sorted(root.rglob("bethesda_ground_truth.json"))
        if matches:
            print(f"Found bethesda_ground_truth.json at: {matches[0]}")
            return matches[0].parent

    raise FileNotFoundError(
        "Не найден bethesda_ground_truth.json. На Kaggle файл обычно лежит в подпапке "
        "вида /kaggle/input/<dataset-slug>/bethesda_ground_truth.json. "
        "Если авто-поиск не сработал, задай os.environ['CYTORAG_DATA_PATH'] с точным путем к файлу."
    )

BASE_DIR = resolve_base_dir()
DATA_PATH = BASE_DIR / "bethesda_ground_truth.json"
EXPERIMENTS_REPORT_PATH = BASE_DIR / "experiments_res.md"
DEFAULT_ARTIFACT_DIR = Path("/kaggle/working/graph_rag_kaggle_experiment") if str(BASE_DIR).startswith("/kaggle/input") else BASE_DIR / "artifacts" / "graph_rag_kaggle_experiment"
ARTIFACT_DIR = Path(os.getenv("GRAPH_RAG_ARTIFACT_DIR", str(DEFAULT_ARTIFACT_DIR)))
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_MODEL_NAME = os.getenv("GRAPH_RAG_EMBEDDING_MODEL", "cointegrated/rubert-tiny2")
RERANKER_NAME = os.getenv("GRAPH_RAG_RERANKER", "DiTy/cross-encoder-russian-msmarco")
LOCAL_LLM_MODEL = os.getenv("GRAPH_RAG_LLM_MODEL", "Qwen/Qwen2.5-3B-Instruct")

RUN_RETRIEVAL = str_to_bool(os.getenv("RUN_RETRIEVAL", "1"), True)
RUN_LOCAL_LLM = str_to_bool(os.getenv("RUN_LOCAL_LLM", "0"), False)
RUN_RAGAS = str_to_bool(os.getenv("RUN_RAGAS", "0"), False)

EMBEDDING_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RERANKER_DEVICE = os.getenv("GRAPH_RAG_RERANKER_DEVICE", "cpu")
LLM_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BASE_TOP_K = int(os.getenv("GRAPH_RAG_BASE_TOP_K", "5"))
GRAPH_EXPANSION_DEPTH = int(os.getenv("GRAPH_RAG_EXPANSION_DEPTH", "2"))
GRAPH_EXPANSION_LIMIT = int(os.getenv("GRAPH_RAG_EXPANSION_LIMIT", "30"))
RERANK_TOP_K = int(os.getenv("GRAPH_RAG_RERANK_TOP_K", "20"))
FINAL_TOP_K = int(os.getenv("GRAPH_RAG_FINAL_TOP_K", "3"))

print({
    "base_dir": str(BASE_DIR),
    "artifacts": str(ARTIFACT_DIR),
    "embedding_model": EMBEDDING_MODEL_NAME,
    "embedding_device": EMBEDDING_DEVICE,
    "reranker": RERANKER_NAME,
    "reranker_device": RERANKER_DEVICE,
    "local_llm_model": LOCAL_LLM_MODEL,
    "run_local_llm": RUN_LOCAL_LLM,
    "run_ragas": RUN_RAGAS,
})


## Plan

1. Load the Bethesda ground-truth dataset.
2. Build three selected chunking views:
   - `sentence_window_2`: best simple retrieval gain.
   - `structural_then_sentence_window_2`: best `faithfulness_ragas` in the prior experiments.
   - `structural_subcase`: best `answer_relevancy` in the prior experiments.
3. For each view, build a graph over cases, chunks, adjacency edges, and shared medical term nodes.
4. Retrieve with dense + BM25, expand over the graph, rerank with a Russian cross-encoder, and evaluate.
5. Optionally generate answers with local Qwen and run RAGAS on Kaggle.

In [ ]:
@dataclass
class ChunkingSpec:
    label: str
    selected_because: str


SELECTED_CHUNKING_STRATEGIES = [
    ChunkingSpec("sentence_window_2", "best simple retrieval strategy: Hit Rate=100, MRR=1.00"),
    ChunkingSpec("structural_then_sentence_window_2", "best faithfulness_ragas in prior experiments"),
    ChunkingSpec("structural_subcase", "best answer_relevancy in prior experiments"),
]


def clear_torch_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def tokenize_ru_medical(text: str) -> List[str]:
    return re.findall(r"[A-Za-zА-Яа-яЁё0-9№/-]+", text.lower())


def extract_case_anchor(text: str, max_len: int = 120) -> str:
    first_sentence = re.split(r"(?<=[.!?])\s+", text.strip())[0]
    return first_sentence[:max_len].rstrip(" ,;:")


def extract_bethesda_label(text: str) -> Optional[str]:
    match = re.search(r"bethesda\s*[-–]\s*([ivx]+)", text, flags=re.IGNORECASE)
    return match.group(1).upper() if match else None


def build_eval_rows(raw_rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    rows = []
    for item in raw_rows:
        anchor = extract_case_anchor(item["simulated_context"])
        rows.append({
            "id": int(item["id"]),
            "original_query": item["query"],
            "retrieval_query": f"Определи диагностическую категорию Bethesda для клинического случая: {anchor}",
            "case_anchor": anchor,
            "simulated_context": item["simulated_context"],
            "ground_truth": item["ground_truth"],
            "bethesda_label": extract_bethesda_label(item["ground_truth"]),
        })
    return rows


raw_rows = json.loads(DATA_PATH.read_text(encoding="utf-8"))
eval_rows = build_eval_rows(raw_rows)

preview_df = pd.DataFrame({
    "id": [row["id"] for row in eval_rows],
    "case_anchor": [row["case_anchor"] for row in eval_rows],
    "bethesda_label": [row["bethesda_label"] for row in eval_rows],
})

print(f"Loaded {len(eval_rows)} Bethesda cases from {DATA_PATH}")
display(preview_df)


In [ ]:
def split_sentences_ru(text: str) -> List[str]:
    sentences = [part.strip() for part in re.split(r"(?<=[.!?])\s+", text.strip()) if part.strip()]
    return sentences or [text.strip()]


def is_subcase_header(sentence: str) -> bool:
    sentence_lc = sentence.lower()
    header_markers = ("пр. доля", "лев. доля", "п ", "л ", "п/п", "н/3", "прав.", "лев.")
    has_case_marker = "№" in sentence
    starts_like_header = any(sentence_lc.startswith(marker) for marker in header_markers)
    if not has_case_marker and not starts_like_header:
        return False
    if len(sentence) > 100 and sentence.count(".") < 2:
        return False
    return starts_like_header or has_case_marker


def split_structural_subcases(text: str) -> List[str]:
    sentences = split_sentences_ru(text)
    segments: List[List[str]] = []
    current: List[str] = []
    for sentence in sentences:
        if is_subcase_header(sentence) and current:
            segments.append(current)
            current = [sentence]
        else:
            current.append(sentence)
    if current:
        segments.append(current)
    return [" ".join(segment).strip() for segment in segments if " ".join(segment).strip()]


def build_sentence_windows(sentences: List[str], window_size: int = 2, overlap: int = 1) -> List[str]:
    if len(sentences) <= window_size:
        return [" ".join(sentences).strip()]
    step = max(1, window_size - overlap)
    windows: List[str] = []
    for start in range(0, len(sentences), step):
        window = sentences[start : start + window_size]
        if not window:
            continue
        windows.append(" ".join(window).strip())
        if start + window_size >= len(sentences):
            break
    return windows


def make_chunk_record(
    text: str,
    row: Dict[str, Any],
    chunking_label: str,
    chunk_index: int,
    parent_segment_index: int = 0,
) -> Tuple[str, Dict[str, Any]]:
    return (
        text,
        {
            "case_id": row["id"],
            "case_anchor": row["case_anchor"],
            "chunking": chunking_label,
            "chunk_id": f"{row['id']}::{chunking_label}::{chunk_index}",
            "chunk_index": chunk_index,
            "parent_segment_index": parent_segment_index,
        },
    )


def build_chunked_documents(rows: List[Dict[str, Any]], spec: ChunkingSpec) -> List[Tuple[str, Dict[str, Any]]]:
    documents: List[Tuple[str, Dict[str, Any]]] = []
    for row in rows:
        text = row["simulated_context"]
        if spec.label == "sentence_window_2":
            windows = build_sentence_windows(split_sentences_ru(text), window_size=2, overlap=1)
            for idx, window in enumerate(windows):
                documents.append(make_chunk_record(window, row, spec.label, idx))
        elif spec.label == "structural_subcase":
            segments = split_structural_subcases(text)
            for idx, segment in enumerate(segments):
                documents.append(make_chunk_record(segment, row, spec.label, idx, parent_segment_index=idx))
        elif spec.label == "structural_then_sentence_window_2":
            chunk_index = 0
            for segment_idx, segment in enumerate(split_structural_subcases(text)):
                windows = build_sentence_windows(split_sentences_ru(segment), window_size=2, overlap=1)
                for window in windows:
                    documents.append(make_chunk_record(window, row, spec.label, chunk_index, parent_segment_index=segment_idx))
                    chunk_index += 1
        else:
            raise ValueError(f"Unsupported chunking strategy: {spec.label}")
    return documents


chunking_preview_df = pd.DataFrame([
    {
        "chunking": spec.label,
        "selected_because": spec.selected_because,
        "indexed_chunks": len(build_chunked_documents(eval_rows, spec)),
        "avg_chunks_per_case": round(len(build_chunked_documents(eval_rows, spec)) / len(eval_rows), 2),
    }
    for spec in SELECTED_CHUNKING_STRATEGIES
])
display(chunking_preview_df)


In [ ]:
STOPWORDS_RU_MED = {
    "на", "и", "в", "с", "по", "для", "как", "из", "их", "при", "что", "это", "или", "не", "нет",
    "фоне", "обнаружено", "обнаружены", "клеток", "клетки", "количества", "количество", "элементы",
    "элементов", "цитограмма", "мазке", "препараты", "щитовидной", "железы", "bethesda",
}


def extract_graph_terms(text: str, max_terms: int = 18) -> List[str]:
    tokens = tokenize_ru_medical(text)
    terms: List[str] = []
    for token in tokens:
        normalized = token.strip("-/").lower()
        if len(normalized) < 4:
            continue
        if normalized in STOPWORDS_RU_MED:
            continue
        if normalized.isdigit():
            continue
        terms.append(normalized)
    counts: Dict[str, int] = {}
    for term in terms:
        counts[term] = counts.get(term, 0) + 1
    ranked = sorted(counts.items(), key=lambda item: (-item[1], item[0]))
    return [term for term, _ in ranked[:max_terms]]


def min_max(values: np.ndarray) -> np.ndarray:
    if len(values) == 0:
        return values
    lo = float(np.min(values))
    hi = float(np.max(values))
    if hi - lo < 1e-9:
        return np.ones_like(values, dtype=np.float32)
    return ((values - lo) / (hi - lo)).astype(np.float32)


class TransformerEmbedder:
    def __init__(self, model_name: str, device: str = EMBEDDING_DEVICE, max_length: int = 384):
        self.model_name = model_name
        self.device = torch.device(device)
        self.max_length = max_length
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()

    def encode(
        self,
        texts: List[str],
        batch_size: int = 32,
        show_progress_bar: bool = False,
        convert_to_numpy: bool = True,
        normalize_embeddings: bool = True,
    ):
        vectors = []
        for start in range(0, len(texts), batch_size):
            batch = texts[start : start + batch_size]
            encoded = self.tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=self.max_length,
                return_tensors="pt",
            ).to(self.device)
            with torch.no_grad():
                outputs = self.model(**encoded)
                token_embeddings = outputs.last_hidden_state
                mask = encoded["attention_mask"].unsqueeze(-1).float()
                pooled = (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
                if normalize_embeddings:
                    pooled = F.normalize(pooled, p=2, dim=1)
            vectors.append(pooled.detach().cpu())
        result = torch.cat(vectors, dim=0)
        return result.numpy() if convert_to_numpy else result


class TransformerCrossEncoder:
    def __init__(self, model_name: str, device: str = RERANKER_DEVICE, max_length: int = 512):
        self.model_name = model_name
        self.device = torch.device(device)
        self.max_length = max_length
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name).to(self.device)
        self.model.eval()

    def predict(self, pairs: List[List[str]], batch_size: int = 16) -> np.ndarray:
        scores = []
        for start in range(0, len(pairs), batch_size):
            batch_pairs = pairs[start : start + batch_size]
            left = [pair[0] for pair in batch_pairs]
            right = [pair[1] for pair in batch_pairs]
            encoded = self.tokenizer(
                left,
                right,
                padding=True,
                truncation=True,
                max_length=self.max_length,
                return_tensors="pt",
            ).to(self.device)
            with torch.no_grad():
                logits = self.model(**encoded).logits
                if logits.shape[-1] == 1:
                    batch_scores = logits[:, 0]
                else:
                    batch_scores = logits[:, -1]
            scores.extend(batch_scores.detach().cpu().float().tolist())
        return np.asarray(scores, dtype=np.float32)


class GraphRAGPipeline:
    def __init__(
        self,
        embedding_model_name: str = EMBEDDING_MODEL_NAME,
        reranker_name: str = RERANKER_NAME,
        embedding_device: str = EMBEDDING_DEVICE,
        reranker_device: str = RERANKER_DEVICE,
    ):
        self.embedding_model_name = embedding_model_name
        self.reranker_name = reranker_name
        self.embedding_device = embedding_device
        self.reranker_device = reranker_device
        self.embedding_model: Optional[TransformerEmbedder] = None
        self.cross_encoder: Optional[TransformerCrossEncoder] = None
        self.index: Optional[faiss.IndexFlatIP] = None
        self.bm25: Optional[BM25Okapi] = None
        self.chunks: List[Dict[str, Any]] = []
        self.graph = nx.Graph()
        self.chunk_node_to_idx: Dict[str, int] = {}

    def load_models(self) -> None:
        print(f"Loading embedding model: {self.embedding_model_name} on {self.embedding_device}")
        self.embedding_model = TransformerEmbedder(self.embedding_model_name, device=self.embedding_device, max_length=384)
        print(f"Loading reranker: {self.reranker_name} on {self.reranker_device}")
        self.cross_encoder = TransformerCrossEncoder(self.reranker_name, device=self.reranker_device, max_length=512)

    def process_documents(self, documents: List[Tuple[str, Dict[str, Any]]]) -> None:
        if self.embedding_model is None or self.cross_encoder is None:
            self.load_models()
        self.chunks = [{"text": text, "metadata": metadata} for text, metadata in documents]
        self._build_graph()
        tokenized_corpus = [tokenize_ru_medical(chunk["text"]) for chunk in self.chunks]
        self.bm25 = BM25Okapi(tokenized_corpus)
        embeddings = self.embedding_model.encode(
            [chunk["text"] for chunk in self.chunks],
            batch_size=32,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        ).astype("float32")
        self.index = faiss.IndexFlatIP(embeddings.shape[1])
        self.index.add(embeddings)
        print(f"Indexed {len(self.chunks)} chunks and built graph with {self.graph.number_of_nodes()} nodes / {self.graph.number_of_edges()} edges")

    def _build_graph(self) -> None:
        self.graph.clear()
        self.chunk_node_to_idx.clear()
        chunks_by_case: Dict[int, List[Tuple[int, Dict[str, Any]]]] = {}
        chunks_by_segment: Dict[Tuple[int, int], List[Tuple[int, Dict[str, Any]]]] = {}
        for idx, chunk in enumerate(self.chunks):
            meta = chunk["metadata"]
            case_node = f"case:{meta['case_id']}"
            chunk_node = f"chunk:{meta['chunk_id']}"
            self.chunk_node_to_idx[chunk_node] = idx
            self.graph.add_node(case_node, kind="case", case_id=meta["case_id"])
            self.graph.add_node(chunk_node, kind="chunk", chunk_idx=idx, case_id=meta["case_id"])
            self.graph.add_edge(case_node, chunk_node, kind="contains", weight=1.0)
            chunks_by_case.setdefault(meta["case_id"], []).append((idx, chunk))
            chunks_by_segment.setdefault((meta["case_id"], meta["parent_segment_index"]), []).append((idx, chunk))
            for term in extract_graph_terms(chunk["text"]):
                term_node = f"term:{term}"
                self.graph.add_node(term_node, kind="term", term=term)
                self.graph.add_edge(chunk_node, term_node, kind="mentions", weight=0.35)
        for _, case_chunks in chunks_by_case.items():
            ordered = sorted(case_chunks, key=lambda pair: pair[1]["metadata"]["chunk_index"])
            for (left_idx, left), (right_idx, right) in zip(ordered, ordered[1:]):
                left_node = f"chunk:{left['metadata']['chunk_id']}"
                right_node = f"chunk:{right['metadata']['chunk_id']}"
                self.graph.add_edge(left_node, right_node, kind="adjacent_case", weight=0.8)
        for _, segment_chunks in chunks_by_segment.items():
            ordered = sorted(segment_chunks, key=lambda pair: pair[1]["metadata"]["chunk_index"])
            for (left_idx, left), (right_idx, right) in zip(ordered, ordered[1:]):
                left_node = f"chunk:{left['metadata']['chunk_id']}"
                right_node = f"chunk:{right['metadata']['chunk_id']}"
                self.graph.add_edge(left_node, right_node, kind="adjacent_segment", weight=1.0)

    def _base_candidates(self, query: str, top_k: int) -> Dict[int, Dict[str, float]]:
        if self.index is None or self.bm25 is None or self.embedding_model is None:
            raise RuntimeError("Index is not built. Call process_documents() first.")
        scores: Dict[int, Dict[str, float]] = {}
        query_embedding = self.embedding_model.encode(
            [query],
            batch_size=1,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        ).astype("float32")
        dense_scores, dense_idx = self.index.search(query_embedding, min(top_k, len(self.chunks)))
        dense_norm = min_max(dense_scores[0])
        for idx, score in zip(dense_idx[0], dense_norm):
            if idx == -1:
                continue
            scores.setdefault(int(idx), {"dense": 0.0, "bm25": 0.0, "graph": 0.0})
            scores[int(idx)]["dense"] = max(scores[int(idx)]["dense"], float(score))
        bm25_scores = self.bm25.get_scores(tokenize_ru_medical(query))
        bm25_idx = np.argsort(bm25_scores)[::-1][: min(top_k, len(self.chunks))]
        bm25_norm = min_max(bm25_scores[bm25_idx].astype("float32"))
        for idx, score in zip(bm25_idx, bm25_norm):
            scores.setdefault(int(idx), {"dense": 0.0, "bm25": 0.0, "graph": 0.0})
            scores[int(idx)]["bm25"] = max(scores[int(idx)]["bm25"], float(score))
        return scores

    def _expand_candidates(self, candidates: Dict[int, Dict[str, float]], depth: int, limit: int) -> None:
        seed_indices = list(candidates.keys())
        for seed_idx in seed_indices:
            seed_chunk = self.chunks[seed_idx]
            seed_node = f"chunk:{seed_chunk['metadata']['chunk_id']}"
            if seed_node not in self.graph:
                continue
            lengths = nx.single_source_shortest_path_length(self.graph, seed_node, cutoff=depth)
            for node, distance in sorted(lengths.items(), key=lambda item: item[1]):
                if not node.startswith("chunk:") or node == seed_node:
                    continue
                expanded_idx = self.chunk_node_to_idx.get(node)
                if expanded_idx is None:
                    continue
                graph_score = 1.0 / (1.0 + float(distance))
                candidates.setdefault(expanded_idx, {"dense": 0.0, "bm25": 0.0, "graph": 0.0})
                candidates[expanded_idx]["graph"] = max(candidates[expanded_idx]["graph"], graph_score)
                if len(candidates) >= limit:
                    return

    def search(
        self,
        query: str,
        base_top_k: int = BASE_TOP_K,
        expansion_depth: int = GRAPH_EXPANSION_DEPTH,
        expansion_limit: int = GRAPH_EXPANSION_LIMIT,
        rerank_top_k: int = RERANK_TOP_K,
        final_top_k: int = FINAL_TOP_K,
    ) -> List[Dict[str, Any]]:
        if self.cross_encoder is None:
            raise RuntimeError("Reranker is not loaded.")
        candidate_scores = self._base_candidates(query, base_top_k)
        self._expand_candidates(candidate_scores, expansion_depth, expansion_limit)
        weighted = []
        for idx, parts in candidate_scores.items():
            score = 0.45 * parts["dense"] + 0.30 * parts["bm25"] + 0.25 * parts["graph"]
            weighted.append((score, idx, parts))
        weighted = sorted(weighted, key=lambda item: item[0], reverse=True)[:rerank_top_k]
        if not weighted:
            return []
        candidate_chunks = [self.chunks[idx] for _, idx, _ in weighted]
        cross_inputs = [[query, item["text"]] for item in candidate_chunks]
        cross_scores = self.cross_encoder.predict(cross_inputs, batch_size=min(16, len(cross_inputs)))
        ranked = sorted(zip(weighted, candidate_chunks, cross_scores), key=lambda item: float(item[2]), reverse=True)
        final_items = []
        for rank, ((fusion_score, idx, parts), chunk, reranker_score) in enumerate(ranked[:final_top_k], start=1):
            final_items.append({
                "rank": rank,
                "text": chunk["text"],
                "metadata": chunk["metadata"],
                "fusion_score": float(fusion_score),
                "dense_score": float(parts["dense"]),
                "bm25_score": float(parts["bm25"]),
                "graph_score": float(parts["graph"]),
                "reranker_score": float(reranker_score),
            })
        return final_items

    @staticmethod
    def build_context_block(retrieved: List[Dict[str, Any]]) -> str:
        blocks = []
        for item in retrieved:
            meta = item["metadata"]
            blocks.append(f"[Case {meta['case_id']} | rank={item['rank']} | chunk={meta['chunk_id']}] {item['text']}")
        return "\n\n---\n\n".join(blocks)

    def cleanup(self) -> None:
        for attr in ["embedding_model", "cross_encoder", "index", "bm25"]:
            setattr(self, attr, None)
        self.chunks = []
        self.graph.clear()
        self.chunk_node_to_idx.clear()
        clear_torch_memory()


In [ ]:
class LocalQwenLLM:
    def __init__(self, model_id: str = LOCAL_LLM_MODEL):
        print(f"Loading local LLM: {model_id} on {LLM_DEVICE}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
        dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=dtype,
            device_map="auto" if torch.cuda.is_available() else None,
            trust_remote_code=True,
        )
        if not torch.cuda.is_available():
            self.model.to("cpu")

    def get_response(self, prompt: str, system_prompt: str = "Ты полезный ассистент.") -> str:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ]
        text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=256,
                temperature=0.1,
                do_sample=False,
                repetition_penalty=1.05,
            )
        response_ids = outputs[0][len(inputs.input_ids[0]) :]
        return self.tokenizer.decode(response_ids, skip_special_tokens=True).strip()

    def as_langchain_pipeline(self):
        text_generation = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
            max_new_tokens=256,
            temperature=0.1,
            do_sample=False,
            return_full_text=False,
        )
        from langchain_huggingface import HuggingFacePipeline

        return HuggingFacePipeline(pipeline=text_generation)


def heuristic_bethesda_answer(context: str) -> str:
    text = context.lower()
    parts = []
    if "клетки эпителия щитовидной железы не найдены" in text or "неинформатив" in text:
        parts.append("Bethesda I: материал неинформативный.")
    if "не представляется возможным" in text and ("небольшое количество" in text or "голых" in text):
        parts.append("Bethesda I: данных недостаточно для уверенной оценки.")
    if "атип" in text and "неясного значения" in text:
        parts.append("Bethesda III: атипия неясного значения.")
    if "фолликулярного новообразования" in text or "трабекул" in text:
        parts.append("Bethesda IV: подозрение на фолликулярное новообразование.")
    if "аденоматозного зоба" in text or "коллоидного зоба" in text or "доброкачествен" in text:
        parts.append("Bethesda II: доброкачественное образование.")
    if not parts:
        return "Я не знаю."
    seen = []
    for part in parts:
        if part not in seen:
            seen.append(part)
    return " ".join(seen)


def generate_answer(query: str, retrieved: List[Dict[str, Any]], llm: Optional[LocalQwenLLM]) -> str:
    context = GraphRAGPipeline.build_context_block(retrieved)
    if llm is None:
        return heuristic_bethesda_answer(context)
    system_prompt = (
        "Ты опытный врач-цитолог. Отвечай только на основе предоставленного контекста. "
        "Сначала назови диагностическую категорию Bethesda, затем дай короткое обоснование. "
        "Если контекста недостаточно, скажи 'Я не знаю'."
    )
    user_prompt = f"Контекст:\n{context}\n\nВопрос: {query}\nОтвет:"
    return llm.get_response(user_prompt, system_prompt)


In [ ]:
def average_precision_at_k(retrieved_case_ids: List[int], gold_case_id: int) -> float:
    hits = 0
    precision_sum = 0.0
    for rank, case_id in enumerate(retrieved_case_ids, start=1):
        if case_id == gold_case_id:
            hits += 1
            precision_sum += hits / rank
    return precision_sum / max(hits, 1) if hits else 0.0


def evaluate_retrieval_rows(rows: List[Dict[str, Any]], label_key: str = "chunking") -> Dict[str, Any]:
    frame = pd.DataFrame(rows)
    label = frame[label_key].iloc[0]
    reciprocal_ranks = []
    context_precision_proxy = []
    context_recall_proxy = []
    for row in rows:
        retrieved_case_ids = row["retrieved_case_ids"]
        gold_id = row["gold_case_id"]
        rank = next((idx + 1 for idx, case_id in enumerate(retrieved_case_ids) if case_id == gold_id), None)
        reciprocal_ranks.append(0.0 if rank is None else 1.0 / rank)
        context_precision_proxy.append(average_precision_at_k(retrieved_case_ids, gold_id))
        context_recall_proxy.append(float(gold_id in retrieved_case_ids))
    return {
        label_key: label,
        "hit_rate_retrieval": round(float(np.mean(frame["gold_case_hit"])) * 100, 2),
        "mrr": round(float(np.mean(reciprocal_ranks)), 4),
        "context_precision_proxy": round(float(np.mean(context_precision_proxy)) * 100, 2),
        "context_recall_proxy": round(float(np.mean(context_recall_proxy)) * 100, 2),
    }


def build_prediction_frame(retrieval_rows: List[Dict[str, Any]], llm: Optional[LocalQwenLLM]) -> pd.DataFrame:
    records = []
    for row in retrieval_rows:
        answer = generate_answer(row["question"], row["retrieved_items"], llm)
        records.append({**row, "answer": answer})
    return pd.DataFrame(records)


def safe_metric_mean(frame: pd.DataFrame, column: str) -> float:
    if column not in frame.columns:
        return np.nan
    series = pd.to_numeric(frame[column], errors="coerce")
    return round(series.mean() * 100, 2) if series.notna().any() else np.nan


def run_ragas_if_enabled(
    chunking_label: str,
    prediction_df: pd.DataFrame,
    llm: Optional[LocalQwenLLM],
) -> Tuple[Dict[str, Any], pd.DataFrame]:
    if not RUN_RAGAS:
        return {
            "chunking": chunking_label,
            "context_precision": np.nan,
            "context_recall": np.nan,
            "faithfulness_ragas": np.nan,
            "answer_relevancy": np.nan,
            "ragas_status": "skipped: RUN_RAGAS=False",
            "ragas_error": "",
        }, pd.DataFrame()
    if llm is None:
        return {
            "chunking": chunking_label,
            "context_precision": np.nan,
            "context_recall": np.nan,
            "faithfulness_ragas": np.nan,
            "answer_relevancy": np.nan,
            "ragas_status": "skipped: RUN_LOCAL_LLM=False",
            "ragas_error": "RAGAS needs generated answers; set RUN_LOCAL_LLM=1 as well as RUN_RAGAS=1.",
        }, pd.DataFrame()
    try:
        from ragas import evaluate
        from ragas.dataset_schema import EvaluationDataset
        from ragas.metrics import answer_relevancy, context_precision, context_recall, faithfulness
        from ragas.run_config import RunConfig
        from langchain_huggingface import HuggingFaceEmbeddings

        records = []
        for row in prediction_df.to_dict(orient="records"):
            records.append({
                "user_input": row["question"],
                "response": row["answer"],
                "retrieved_contexts": row["retrieved_contexts"],
                "reference": row["ground_truth"],
            })
        dataset = EvaluationDataset.from_list(records)
        ragas_embeddings = HuggingFaceEmbeddings(
            model_name=EMBEDDING_MODEL_NAME,
            model_kwargs={"device": "cpu"},
            encode_kwargs={"normalize_embeddings": True},
        )
        result = evaluate(
            dataset=dataset,
            metrics=[context_precision, context_recall, faithfulness, answer_relevancy],
            llm=llm.as_langchain_pipeline(),
            embeddings=ragas_embeddings,
            run_config=RunConfig(max_workers=1, timeout=240),
            raise_exceptions=False,
        )
        detail_df = result.to_pandas()
        return {
            "chunking": chunking_label,
            "context_precision": safe_metric_mean(detail_df, "context_precision"),
            "context_recall": safe_metric_mean(detail_df, "context_recall"),
            "faithfulness_ragas": safe_metric_mean(detail_df, "faithfulness"),
            "answer_relevancy": safe_metric_mean(detail_df, "answer_relevancy"),
            "ragas_status": "ok",
            "ragas_error": "",
        }, detail_df
    except Exception as exc:
        error_text = f"{type(exc).__name__}: {exc}"
        print(f"RAGAS failed for {chunking_label}: {error_text}")
        return {
            "chunking": chunking_label,
            "context_precision": np.nan,
            "context_recall": np.nan,
            "faithfulness_ragas": np.nan,
            "answer_relevancy": np.nan,
            "ragas_status": "failed",
            "ragas_error": error_text,
        }, pd.DataFrame()


In [ ]:
retrieval_summary_rows: List[Dict[str, Any]] = []
ragas_summary_rows: List[Dict[str, Any]] = []
all_prediction_frames: Dict[str, pd.DataFrame] = {}
all_retrieval_payload: Dict[str, Any] = {}

llm = LocalQwenLLM() if RUN_LOCAL_LLM else None

if RUN_RETRIEVAL:
    for spec in SELECTED_CHUNKING_STRATEGIES:
        print("=" * 90)
        print(f"Graph RAG run: {spec.label}")
        pipeline_obj: Optional[GraphRAGPipeline] = None
        try:
            clear_torch_memory()
            chunked_documents = build_chunked_documents(eval_rows, spec)
            pipeline_obj = GraphRAGPipeline()
            pipeline_obj.process_documents(chunked_documents)

            rows: List[Dict[str, Any]] = []
            for row in eval_rows:
                retrieved = pipeline_obj.search(row["retrieval_query"])
                retrieved_case_ids = [item["metadata"]["case_id"] for item in retrieved]
                retrieved_chunk_ids = [item["metadata"]["chunk_id"] for item in retrieved]
                retrieved_contexts = [item["text"] for item in retrieved]
                rank = next((idx + 1 for idx, case_id in enumerate(retrieved_case_ids) if case_id == row["id"]), None)
                rows.append({
                    "chunking": spec.label,
                    "selected_because": spec.selected_because,
                    "hf_model": EMBEDDING_MODEL_NAME,
                    "reranker": RERANKER_NAME,
                    "query_id": row["id"],
                    "question": row["retrieval_query"],
                    "case_anchor": row["case_anchor"],
                    "ground_truth": row["ground_truth"],
                    "bethesda_label": row["bethesda_label"],
                    "gold_case_id": row["id"],
                    "retrieved_case_ids": retrieved_case_ids,
                    "retrieved_chunk_ids": retrieved_chunk_ids,
                    "retrieved_contexts": retrieved_contexts,
                    "retrieved_items": retrieved,
                    "gold_case_hit": int(row["id"] in retrieved_case_ids),
                    "gold_case_rank": rank,
                })

            retrieval_summary = evaluate_retrieval_rows(rows, label_key="chunking")
            retrieval_summary.update({
                "hf_model": EMBEDDING_MODEL_NAME,
                "reranker": RERANKER_NAME,
                "indexed_chunks": len(chunked_documents),
                "avg_chunks_per_case": round(len(chunked_documents) / len(eval_rows), 2),
                "selected_because": spec.selected_because,
                "status": "ok",
            })
            retrieval_summary_rows.append(retrieval_summary)

            prediction_df = build_prediction_frame(rows, llm)
            all_prediction_frames[spec.label] = prediction_df
            ragas_summary, ragas_detail_df = run_ragas_if_enabled(spec.label, prediction_df, llm)
            ragas_summary_rows.append(ragas_summary)

            prediction_path = ARTIFACT_DIR / f"{spec.label}_predictions.csv"
            detail_path = ARTIFACT_DIR / f"{spec.label}_ragas_details.csv"
            prediction_df.drop(columns=["retrieved_items"], errors="ignore").to_csv(prediction_path, index=False, encoding="utf-8-sig")
            if not ragas_detail_df.empty:
                ragas_detail_df.to_csv(detail_path, index=False, encoding="utf-8-sig")

            all_retrieval_payload[spec.label] = {
                "status": "ok",
                "spec": asdict(spec),
                "indexed_chunks": len(chunked_documents),
                "rows": [
                    {key: value for key, value in item.items() if key != "retrieved_items"}
                    for item in rows
                ],
            }
        except Exception as exc:
            retrieval_summary_rows.append({
                "chunking": spec.label,
                "hf_model": EMBEDDING_MODEL_NAME,
                "status": "failed",
                "error": f"{type(exc).__name__}: {exc}",
            })
            ragas_summary_rows.append({
                "chunking": spec.label,
                "ragas_status": "skipped after retrieval failure",
            })
            all_retrieval_payload[spec.label] = {
                "status": "failed",
                "spec": asdict(spec),
                "error": f"{type(exc).__name__}: {exc}",
                "rows": [],
            }
        finally:
            if pipeline_obj is not None:
                pipeline_obj.cleanup()

retrieval_summary_df = pd.DataFrame(retrieval_summary_rows)
ragas_summary_df = pd.DataFrame(ragas_summary_rows)

if not retrieval_summary_df.empty:
    retrieval_summary_df.to_csv(ARTIFACT_DIR / "graph_rag_retrieval_summary.csv", index=False, encoding="utf-8-sig")
if not ragas_summary_df.empty:
    ragas_summary_df.to_csv(ARTIFACT_DIR / "graph_rag_ragas_summary.csv", index=False, encoding="utf-8-sig")
(ARTIFACT_DIR / "graph_rag_retrieval_payload.json").write_text(
    json.dumps(all_retrieval_payload, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

display(retrieval_summary_df)
display(ragas_summary_df)


In [ ]:
comparison_df = retrieval_summary_df.copy()
if not comparison_df.empty and not ragas_summary_df.empty:
    comparison_df = comparison_df.merge(ragas_summary_df, on="chunking", how="left")

metric_columns = [
    "chunking",
    "hit_rate_retrieval",
    "mrr",
    "context_precision_proxy",
    "context_recall_proxy",
    "context_precision",
    "context_recall",
    "faithfulness_ragas",
    "answer_relevancy",
    "indexed_chunks",
    "ragas_status",
    "ragas_error",
    "selected_because",
]
metric_columns = [column for column in metric_columns if column in comparison_df.columns]
comparison_df = comparison_df[metric_columns] if not comparison_df.empty else comparison_df
comparison_path = ARTIFACT_DIR / "graph_rag_combined_metrics.csv"
comparison_df.to_csv(comparison_path, index=False, encoding="utf-8-sig")
display(comparison_df)


def dataframe_to_markdown(frame: pd.DataFrame) -> str:
    if frame.empty:
        return "_No rows to display._"
    view = frame.copy()
    for column in view.columns:
        if pd.api.types.is_float_dtype(view[column]):
            view[column] = view[column].map(lambda value: "" if pd.isna(value) else f"{value:.2f}")
        else:
            view[column] = view[column].map(lambda value: "" if pd.isna(value) else str(value))
    headers = list(view.columns)
    rows = view.to_numpy().tolist()
    widths = [len(header) for header in headers]
    for row in rows:
        widths = [max(width, len(cell)) for width, cell in zip(widths, row)]
    def fmt(row: Iterable[str]) -> str:
        return "| " + " | ".join(str(cell).ljust(width) for cell, width in zip(row, widths)) + " |"
    lines = [fmt(headers), "| " + " | ".join("-" * width for width in widths) + " |"]
    lines.extend(fmt(row) for row in rows)
    return "\n".join(lines)


best_line = "- No completed Graph RAG runs."
if not comparison_df.empty and "hit_rate_retrieval" in comparison_df.columns:
    ranked = comparison_df.sort_values(["hit_rate_retrieval", "mrr"], ascending=[False, False], na_position="last")
    if not ranked.empty:
        best = ranked.iloc[0]
        best_line = f"- Best retrieval run: `{best['chunking']}` with Hit Rate `{best.get('hit_rate_retrieval', np.nan)}` and MRR `{best.get('mrr', np.nan)}`."

report_lines = [
    "# CytoRAG Graph RAG Kaggle Experiment",
    "",
    "## Configuration",
    "",
    f"- Embedding model: `{EMBEDDING_MODEL_NAME}`",
    f"- Reranker: `{RERANKER_NAME}`",
    f"- Local LLM: `{LOCAL_LLM_MODEL}`",
    f"- RUN_LOCAL_LLM: `{RUN_LOCAL_LLM}`",
    f"- RUN_RAGAS: `{RUN_RAGAS}`",
    f"- Dataset: `{DATA_PATH}`",
    "",
    "## Metrics",
    "",
    dataframe_to_markdown(comparison_df),
    "",
    "## Notes",
    "",
    best_line,
    "- `context_precision_proxy` and `context_recall_proxy` are deterministic retrieval proxies based on whether retrieved chunks belong to the gold case.",
    "- `context_precision`, `context_recall`, `faithfulness_ragas`, and `answer_relevancy` are filled only when `RUN_RAGAS=True`.",
]
report_path = ARTIFACT_DIR / "graph_rag_report.md"
report_path.write_text("\n".join(report_lines), encoding="utf-8")
print(f"Combined metrics saved to: {comparison_path}")
print(f"Markdown report saved to: {report_path}")


## Validation Notes

- Fast retrieval validation: run with defaults (`RUN_LOCAL_LLM=0`, `RUN_RAGAS=0`). This produces retrieval metrics and proxy context metrics.
- Full Kaggle evaluation: set environment variables or edit the config cell:
  - `RUN_LOCAL_LLM=1`
  - `RUN_RAGAS=1`
- Expected artifacts:
  - `graph_rag_retrieval_summary.csv`
  - `graph_rag_ragas_summary.csv`
  - `graph_rag_combined_metrics.csv`
  - `<chunking>_predictions.csv`
  - `graph_rag_report.md`

This notebook does not modify `RAG_local.ipynb`, `experiments_res.md`, or previous experiment outputs.